# scRareBench | High-Level scDisInFact | Dataset 4 — COVID-19 autoimmunity PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

High-level flow:

`load_dataset(4) → MethodSpec(scDisInFact) → benchmark_method() → scIB + rare benchmark → HTML / multi-seed ZIP`

Pinned revisions:

- scRareBench: `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`
- scDisInFact: `9466ecf257d923f879cb39712c530d24826d88bc`

At notebook creation time, scRareBench GitHub `main` is identical to the pinned commit.

## Dataset 4 contract

- dataset: `covid19_autoimmunity_pbmc`
- 97,499 cells × 21,858 features
- biological label: `cell_type` — **evaluation only**
- integration/evaluation batch: `donor_id`
- canonical raw counts: `layers["counts"]`
- 22 donors
- 26 biological label categories
- scIB HVG mode: `evaluation_batch`
- registered rare-scenario metadata: package-owned

## scDisInFact policy for Dataset 4

scDisInFact is designed for multi-batch/multi-condition data. The current
scRareBench Dataset 4 contract defines `donor_id` as the integration batch but
does **not** define a validated independent method condition.

To avoid injecting an arbitrary disease/group covariate or leaking the benchmark
ground-truth label, this notebook uses one method-only constant condition:

`baseline`

The evaluated representation is only the batch-corrected shared factor:

`mu_c`

Method-side HVG policy:

- donor-aware Seurat-v3
- 4,000 HVGs
- `batch_key="donor_id"`
- `span=0.3`

The Dataset 4 audit showed donor-aware HVG selection is numerically valid, so the
Dataset-3 global-HVG workaround is not used here.

Other method settings remain aligned with the Dataset 3 scDisInFact notebook:

- `Ks=[8, 4]`
- batch size 64
- 100 epochs
- NB reconstruction
- shared latent dimension 8
- final representation key `X_scdisinfact_shared`

Dataset preparation, scenario annotation, scIB reference preprocessing,
evaluation, reporting, and multi-seed delivery remain package-owned.

Use **GPU + High-RAM Colab**. scDisInFact densifies data internally.

For a first smoke run, temporarily set:

`METHOD_SEEDS = [42]`

## 1. Install pinned scRareBench and scDisInFact

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDISINFACT_COMMIT = "9466ecf257d923f879cb39712c530d24826d88bc"

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)
SCDISINFACT_GITHUB = (
    "git+https://github.com/ZhangLabGT/"
    f"scDisInFact.git@{SCDISINFACT_COMMIT}"
)

# Bootstrap the exact benchmark package revision first.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_RUNTIME_DEPENDENCIES = (
    "torch>=1.11.0",
    "scikit-learn>=0.22.1",
    "matplotlib>=3.5.2",
    "scikit-misc>=0.3",
)

from scrarebench.runtime import setup_runtime, print_install_report

report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=("torch", "sklearn", "matplotlib", "skmisc"),
    quiet=False,
)
print_install_report(report)

# Install the exact scDisInFact source revision without its legacy dependency pins.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    SCDISINFACT_GITHUB,
])

# Fresh-process method import check.
subprocess.check_call([
    sys.executable,
    "-c",
    (
        "import scDisInFact, torch; "
        "from scDisInFact import create_scdisinfact_dataset, scdisinfact; "
        "print('scDisInFact', getattr(scDisInFact, '__version__', 'unknown')); "
        "print('CUDA', torch.cuda.is_available())"
    ),
])


## 2. Load Dataset 4 through scRareBench

In [ ]:
import torch
import scrarebench
from scrarebench import (
    dataset_info,
    load_dataset,
    scenario_table_from_adata,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. In Colab select Runtime → Change runtime type → GPU."
    )

adata = load_dataset(4)
info = dataset_info(adata)

# Fail-fast package-contract validation only.
expected = {
    "dataset_key": "covid19_autoimmunity_pbmc",
    "label_key": "cell_type",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "evaluation_batch",
    "benchmark_ready": True,
}

for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Dataset 4 contract mismatch for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (97_499, 21_858):
    raise RuntimeError(
        f"Unexpected Dataset 4 shape: {adata.shape}; "
        "expected (97499, 21858)."
    )

if "counts" not in adata.layers:
    raise RuntimeError(
        "Dataset 4 canonical raw-count layer 'counts' is missing."
    )

if not adata.obs_names.is_unique:
    raise RuntimeError(
        "Dataset 4 cell identifiers are not unique."
    )

if not adata.var_names.is_unique:
    raise RuntimeError(
        "Dataset 4 feature identifiers are not unique."
    )

if adata.obs["donor_id"].astype(str).nunique() != 22:
    raise RuntimeError(
        "Unexpected Dataset 4 donor cardinality; expected 22."
    )

if adata.obs["cell_type"].astype(str).nunique() != 26:
    raise RuntimeError(
        "Unexpected Dataset 4 label cardinality; expected 26."
    )

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or scenario_table.empty:
    raise RuntimeError(
        "Dataset 4 registered six-state scenario metadata is missing."
    )

if len(scenario_table) != 12:
    raise RuntimeError(
        f"Unexpected Dataset 4 scenario-table size: {len(scenario_table)}; "
        "expected 12."
    )

print("scRareBench:", scrarebench.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(adata)
print("\nDataset contract:")
for key in (
    "dataset_key",
    "label_key",
    "batch_key",
    "count_layer",
    "scib_hvg_batch_mode",
    "benchmark_ready",
):
    print(f"  {key}: {info.get(key)}")

print("Donors:", adata.obs["donor_id"].astype(str).nunique())
print("Cell-type categories:", adata.obs["cell_type"].astype(str).nunique())
print("Registered six-state rows:", len(scenario_table))


## 3. scDisInFact configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Dataset 4 validated method-side HVG policy.
    "n_hvg": 4000,
    "hvg_policy": "donor_aware_seurat_v3_raw_counts",
    "hvg_batch_key": "donor_id",
    "hvg_span": 0.3,

    # No validated independent method condition exists in the package contract.
    "condition_column": "_scdisinfact_condition",
    "condition_value": "baseline",

    # scDisInFact architecture/training settings.
    "Ks": [8, 4],
    "batch_size": 64,
    "interval": 10,
    "lr": 5e-4,
    "reg_mmd_comm": 1e-4,
    "reg_mmd_diff": 1e-4,
    "reg_gl": 1.0,
    "reg_class": 1.0,
    "reg_kl_comm": 1e-5,
    "reg_kl_diff": 1e-2,
    "enc_injection": False,
    "nepochs": 100,
    "recon_loss": "NB",
}

OUTPUT_DIR = "/content/scrarebench_scdisinfact_dataset4_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdisinfact_dataset4_method_runs"


## 4. User-owned scDisInFact runner

This is the only method-specific section.

The runner performs only:

1. donor-aware Seurat-v3 HVG selection from canonical raw counts,
2. creation of a constant method-only `baseline` condition,
3. scDisInFact dataset construction/training,
4. extraction of the shared `mu_c` factor,
5. exact restoration of the original cell order.

No benchmark biological label is referenced by the runner.


In [ ]:
from pathlib import Path
import gc
import json
import random
import shutil

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from scDisInFact import create_scdisinfact_dataset, scdisinfact
from scrarebench import MethodOutput, dataset_info

_BARCODE_COL = "_scrarebench_original_barcode"


def _count_guard(matrix, seed, max_values=100_000):
    values = (
        matrix.data
        if sp.issparse(matrix)
        else np.asarray(matrix).reshape(-1)
    )
    values = np.asarray(values)

    if values.size > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[
            rng.choice(
                values.size,
                size=int(max_values),
                replace=False,
            )
        ]

    values = values.astype(float, copy=False)

    if values.size == 0:
        return

    if not np.isfinite(values).all():
        raise ValueError(
            "Canonical counts contain NaN or infinite values."
        )

    if float(values.min()) < 0:
        raise ValueError(
            "Canonical counts contain negative values."
        )

    if not np.allclose(
        values,
        np.rint(values),
        atol=1e-6,
        rtol=0.0,
    ):
        raise ValueError(
            "Canonical counts are not integer-like."
        )


def run_scdisinfact(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "covid19_autoimmunity_pbmc":
        raise RuntimeError(
            "This runner is configured only for scRareBench Dataset 4."
        )

    if batch_key != "donor_id":
        raise RuntimeError(
            f"Unexpected Dataset 4 batch key {batch_key!r}; "
            "expected 'donor_id'."
        )

    if method_info.get("scib_hvg_batch_mode") != "evaluation_batch":
        raise RuntimeError(
            "Dataset 4 is expected to use evaluation-batch-aware HVGs."
        )

    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(
            f"Missing canonical count layer {count_layer!r}."
        )

    if batch_key not in method_adata.obs.columns:
        raise KeyError(
            f"Missing batch column {batch_key!r}."
        )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required for this Dataset 4 scDisInFact run."
        )

    hvg_batch_key = str(config["hvg_batch_key"])
    if hvg_batch_key != batch_key:
        raise RuntimeError(
            f"Dataset 4 method-side HVG batch must match donor_id: "
            f"hvg_batch_key={hvg_batch_key!r}, batch_key={batch_key!r}."
        )

    original_order = (
        method_adata.obs_names
        .astype(str)
        .to_numpy(copy=True)
    )

    if len(np.unique(original_order)) != len(original_order):
        raise RuntimeError(
            "Benchmark cell identifiers must be unique."
        )

    _count_guard(
        method_adata.layers[count_layer],
        seed,
    )

    # Dataset 4 validated donor-aware HVG policy.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(
            int(config["n_hvg"]),
            method_adata.n_vars,
        ),
        batch_key=hvg_batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    if "highly_variable" not in method_adata.var.columns:
        raise RuntimeError(
            "Seurat-v3 HVG selection did not return a mask."
        )

    hvg_mask = (
        method_adata.var["highly_variable"]
        .fillna(False)
        .to_numpy(dtype=bool)
    )

    n_selected = int(hvg_mask.sum())
    expected_hvg = min(
        int(config["n_hvg"]),
        method_adata.n_vars,
    )

    if n_selected != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} donor-aware HVGs, "
            f"selected {n_selected}."
        )

    counts_hvg = (
        method_adata.layers[count_layer][:, hvg_mask]
    )

    counts_hvg = (
        counts_hvg.tocsr().astype(np.float32)
        if sp.issparse(counts_hvg)
        else np.asarray(
            counts_hvg,
            dtype=np.float32,
        )
    )

    # scDisInFact normalizes internally, but every cell must retain expression
    # in the selected feature space.
    cell_sums = np.asarray(
        counts_hvg.sum(axis=1)
    ).reshape(-1)

    zero_cells = int(
        np.count_nonzero(cell_sums <= 0)
    )

    if zero_cells:
        raise ValueError(
            f"{zero_cells} cells have zero counts across selected HVGs."
        )

    condition_col = str(
        config["condition_column"]
    )
    condition_value = str(
        config["condition_value"]
    )

    # Method metadata contains only:
    # - donor batch
    # - constant method-only condition
    # - original barcode guard
    # No biological benchmark label is copied here.
    meta_cells = pd.DataFrame(
        index=method_adata.obs_names.copy()
    )

    meta_cells[batch_key] = (
        method_adata.obs[batch_key]
        .astype(str)
        .to_numpy()
    )

    meta_cells[condition_col] = condition_value
    meta_cells[_BARCODE_COL] = original_order

    if meta_cells[condition_col].nunique() != 1:
        raise RuntimeError(
            "The method-only scDisInFact condition must be constant."
        )

    if meta_cells[batch_key].nunique() != 22:
        raise RuntimeError(
            "Unexpected donor cardinality inside method metadata."
        )

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))

    device = torch.device("cuda:0")

    seed_dir = (
        Path(METHOD_WORK_DIR)
        / f"seed_{int(seed)}"
    )

    if seed_dir.exists():
        shutil.rmtree(seed_dir)

    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        f"Seed {seed}: "
        f"cells={method_adata.n_obs:,}; "
        f"HVGs={n_selected:,}; "
        f"donors={meta_cells[batch_key].nunique()}; "
        f"condition={condition_value!r}; "
        f"GPU={torch.cuda.get_device_name(0)}"
    )

    data_dict = create_scdisinfact_dataset(
        counts_hvg,
        meta_cells,
        condition_key=[condition_col],
        batch_key=batch_key,
        batch_cond_key=None,
        log_trans=True,
    )

    model = scdisinfact(
        data_dict=data_dict,
        reg_mmd_comm=float(
            config["reg_mmd_comm"]
        ),
        reg_mmd_diff=float(
            config["reg_mmd_diff"]
        ),
        reg_gl=float(
            config["reg_gl"]
        ),
        reg_class=float(
            config["reg_class"]
        ),
        reg_kl_comm=float(
            config["reg_kl_comm"]
        ),
        reg_kl_diff=float(
            config["reg_kl_diff"]
        ),
        Ks=list(config["Ks"]),
        batch_size=int(
            config["batch_size"]
        ),
        interval=int(
            config["interval"]
        ),
        lr=float(
            config["lr"]
        ),
        seed=int(seed),
        enc_injection=bool(
            config["enc_injection"]
        ),
        device=device,
    )

    losses = model.train_model(
        nepochs=int(
            config["nepochs"]
        ),
        recon_loss=str(
            config["recon_loss"]
        ),
    )

    model.eval()

    latent_parts = []
    barcode_parts = []

    with torch.no_grad():
        for dataset_part, meta_part in zip(
            data_dict["datasets"],
            data_dict["meta_cells"],
        ):
            inf = model.inference(
                counts=dataset_part.counts_norm.to(
                    model.device
                ),
                batch_ids=dataset_part.batch_id[
                    :, None
                ].to(
                    model.device
                ),
                print_stat=False,
            )

            latent_parts.append(
                inf["mu_c"]
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            barcode_parts.append(
                meta_part[_BARCODE_COL]
                .astype(str)
                .to_numpy()
            )

    latent_native = np.concatenate(
        latent_parts,
        axis=0,
    )

    native_barcodes = np.concatenate(
        barcode_parts,
        axis=0,
    )

    expected_shape = (
        method_adata.n_obs,
        int(config["Ks"][0]),
    )

    if latent_native.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected shared latent shape: {latent_native.shape}; "
            f"expected {expected_shape}."
        )

    if not np.isfinite(
        latent_native
    ).all():
        raise ValueError(
            "Shared mu_c latent contains NaN or infinite values."
        )

    if len(
        np.unique(native_barcodes)
    ) != len(native_barcodes):
        raise RuntimeError(
            "Recovered cell identifiers are not unique."
        )

    if set(native_barcodes) != set(original_order):
        raise RuntimeError(
            "scDisInFact changed the Dataset 4 cell set."
        )

    # Restore exact original scRareBench row order.
    position = pd.Series(
        np.arange(
            len(native_barcodes)
        ),
        index=native_barcodes,
    )

    reorder = (
        position.loc[
            original_order
        ]
        .to_numpy()
    )

    latent = latent_native[
        reorder
    ]

    if not np.array_equal(
        native_barcodes[reorder],
        original_order,
    ):
        raise RuntimeError(
            "Failed to restore the original Dataset 4 cell order."
        )

    if not np.isfinite(latent).all():
        raise ValueError(
            "Aligned shared mu_c latent contains NaN or infinite values."
        )

    # Save lightweight method provenance.
    loss_names = [
        "total",
        "reconstruction",
        "kl_common",
        "kl_diff",
        "mmd_common",
        "mmd_diff",
        "classification",
        "group_lasso_diff",
    ]

    loss_path = (
        seed_dir
        / "training_validation_losses.csv"
    )

    pd.DataFrame({
        name: values
        for name, values
        in zip(loss_names, losses)
    }).to_csv(
        loss_path,
        index=False,
    )

    config_path = (
        seed_dir
        / "method_config.json"
    )

    payload = dict(config)

    payload.update({
        "seed": int(seed),
        "selected_hvgs": n_selected,
        "upstream_commit": SCDISINFACT_COMMIT,
        "condition_policy": (
            "constant baseline; no independent condition "
            "defined by Dataset 4 benchmark contract"
        ),
        "evaluated_factor": "mu_c",
        "evaluated_dimension": int(
            config["Ks"][0]
        ),
    })

    config_path.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Seed {seed}: "
        f"aligned shared latent={latent.shape}"
    )

    del (
        model,
        data_dict,
        latent_native,
        counts_hvg,
        position,
        reorder,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdisinfact_shared",
        provenance_files={
            "scdisinfact_training_losses": loss_path,
            "scdisinfact_method_config": config_path,
        },
    )


## 5. Run the high-level multi-seed benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDisInFact",
    runner=run_scdisinfact,
    config=METHOD_CONFIG,
    dependencies=(
        METHOD_RUNTIME_DEPENDENCIES
        + (SCDISINFACT_GITHUB,)
    ),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,

        # Dataset 4 package metadata supplies:
        # scib_hvg_batch_mode="evaluation_batch".
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,

    # Each seed receives an isolated method input.
    copy_input_per_seed=True,
)


## 6. Inspect results and alignment

In [ ]:
from IPython.display import display

display(
    result.summary().round(5)
)

print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print(
        "  representation:",
        run.representation_key,
    )
    print(
        "  alignment:",
        run.alignment,
    )
    print(
        "  report:",
        run.interactive_report_path,
    )
    print(
        "  bundle:",
        run.bundle_path,
    )

    if (
        run.representation_key
        != "X_scdisinfact_shared"
    ):
        raise RuntimeError(
            f"Seed {seed} returned an unexpected representation key."
        )

    if (
        run.alignment.get(
            "exact_order_match"
        )
        is not True
    ):
        raise RuntimeError(
            f"Seed {seed} did not preserve exact barcode order."
        )


## 7. Download final multi-seed archive

In [ ]:
from pathlib import Path

if result.archive_path is None:
    raise RuntimeError(
        "No final archive was created. "
        "Run all final seeds with finalize=True."
    )

archive = Path(
    result.archive_path
)

if not archive.exists():
    raise FileNotFoundError(
        archive
    )

print(
    "Archive:",
    archive,
)

print(
    "Size (MB):",
    round(
        archive.stat().st_size
        / 1024
        / 1024,
        2,
    ),
)

# Copy to a short Colab path before triggering browser download.
# This avoids some frontend issues with long generated paths/names.
download_copy = Path(
    "/content/scDisInFact_dataset4_results.zip"
)

if (
    archive.resolve()
    != download_copy.resolve()
):
    shutil.copy2(
        archive,
        download_copy,
    )

print(
    "Download copy:",
    download_copy,
)

try:
    from google.colab import files
    files.download(
        str(download_copy)
    )
except ImportError:
    print(
        "Not running in Colab; archive remains at:",
        download_copy,
    )


## Runtime note

For the first Colab execution, temporarily use:

```python
METHOD_SEEDS = [42]
```

After the smoke run succeeds, restore:

```python
METHOD_SEEDS = [42, 123, 2026]
```

for the final benchmark and multi-seed archive.

This notebook is **statically validated**. It does not claim successful runtime
execution until Dataset 4 + scDisInFact actually completes in Colab.
